In [1]:
pip install newsapi-python

Note: you may need to restart the kernel to use updated packages.


In [2]:
topic = input("Filter by topic, what type of news would you like:")

Filter by topic, what type of news would you like:ai


In [3]:
from newsapi import NewsApiClient

# Init
newsapi = NewsApiClient(api_key='a706251e62f54b20a5c71f8618d12a2a')

# /v2/top-headlines
top_headlines = newsapi.get_top_headlines(q=topic,
                                          category='technology',
                                          language='en',
                                          country='us',
                                          page=1)
#sources='bbc-news,the-verge',

In [4]:
# List of URLs to filter out
urls_to_exclude = ['youtube.com', 'androidpolice.com']

# Filter out articles based on the list of URLs
filtered_top_headlines = {
    'articles': [article for article in top_headlines['articles'] if all(exclude_url not in article['url'] for exclude_url in urls_to_exclude)]
}

In [5]:
filtered_top_headlines

{'articles': [{'source': {'id': 'fox-news', 'name': 'Fox News'},
   'author': 'Kurt Knutsson, CyberGuy Report',
   'title': 'Honda’s Uni-One unleashes experience of floating in air without ever leaving ground - Fox News',
   'description': "Honda's Uni-One is a hands-free electric mobility device that can reach speeds of 3.7 mph and support a user weighing up to 242 pounds.",
   'url': 'https://www.foxnews.com/tech/hondas-uni-one-unleashes-experience-floating-air-without-ever-leaving-ground',
   'urlToImage': 'https://static.foxnews.com/foxnews.com/content/uploads/2024/03/6-Honda’s-Uni-One-unleashes-the-experience-of-floating-in-the-air-without-ever-leaving-the-ground.jpg',
   'publishedAt': '2024-03-13T10:00:00Z',
   'content': 'Join Fox News for access to this content\r\nPlus special access to select articles and other premium content with your account - free of charge.\r\nPlease enter a valid email address.\r\nBy entering your e… [+3962 chars]'},
  {'source': {'id': None, 'name': 'P

In [6]:
top_headlines

{'status': 'ok',
 'totalResults': 18,
 'articles': [{'source': {'id': 'fox-news', 'name': 'Fox News'},
   'author': 'Kurt Knutsson, CyberGuy Report',
   'title': 'Honda’s Uni-One unleashes experience of floating in air without ever leaving ground - Fox News',
   'description': "Honda's Uni-One is a hands-free electric mobility device that can reach speeds of 3.7 mph and support a user weighing up to 242 pounds.",
   'url': 'https://www.foxnews.com/tech/hondas-uni-one-unleashes-experience-floating-air-without-ever-leaving-ground',
   'urlToImage': 'https://static.foxnews.com/foxnews.com/content/uploads/2024/03/6-Honda’s-Uni-One-unleashes-the-experience-of-floating-in-the-air-without-ever-leaving-the-ground.jpg',
   'publishedAt': '2024-03-13T10:00:00Z',
   'content': 'Join Fox News for access to this content\r\nPlus special access to select articles and other premium content with your account - free of charge.\r\nPlease enter a valid email address.\r\nBy entering your e… [+3962 chars]'}

Once we have all the URLs, we will have to compare this text with the text that is provided to the model. This will help us create a recommendation system. We will score each new piece on a similarity test and give the highest one as a recommendation.

In [7]:
Searched_article_urls = [article['url'] for article in filtered_top_headlines['articles']]

In [8]:
Searched_article_urls

['https://www.foxnews.com/tech/hondas-uni-one-unleashes-experience-floating-air-without-ever-leaving-ground',
 'https://petapixel.com/2024/03/12/nikon-updates-z9-again-even-better-for-portraits-wildlife-and-sports/',
 'https://www.businessinsider.com/meta-headset-inception-attacks-trap-users-fake-environment-study-2024-3',
 'https://www.xda-developers.com/copilot-gpt-4-turbo-model-free/',
 'https://www.zdnet.com/home-and-office/networking/at-35-the-web-is-broken-but-its-inventor-hasnt-given-up-hope-of-fixing-it/',
 'https://www.searchenginejournal.com/googles-march-2024-core-update-impact-hundreds-of-websites-deindexed/510981/',
 'https://arstechnica.com/tech-policy/2024/03/nyt-disputes-openai-hacking-claim-by-pointing-to-chatgpt-bypassing-paywalls/',
 'https://www.ign.com/articles/helldivers-2-should-feel-like-when-you-are-going-into-a-dark-basement-dev-says',
 'https://www.bleepingcomputer.com/',
 'https://www.macrumors.com/2024/03/12/low-price-15-inch-m3-macbook-air/',
 'https://www

Getting the text from the news articles

In [9]:
import nltk


In [10]:
pip install newspaper3k

Note: you may need to restart the kernel to use updated packages.


In [11]:
from newspaper import Article, ArticleException

In [12]:
nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\js774\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [13]:
def scrape_articles(article_urls):
    articles = []
    failed_articles = []

    for url in article_urls:
        try:
            article = Article(url)
            article.download()
            article.parse()
            article.nlp()
            articles.append(article)
        except ArticleException as e:
            print(f"Error processing article at {url}: {e}")
            failed_articles.append(url)

    return articles, failed_articles

In [14]:
searched_articles_objs, failed_searched_articles = scrape_articles(Searched_article_urls)

Error processing article at https://www.xda-developers.com/copilot-gpt-4-turbo-model-free/: Article `download()` failed with ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')) on URL https://www.xda-developers.com/copilot-gpt-4-turbo-model-free/
Error processing article at https://www.makeuseof.com/generative-ai-tools-improve-work-life/: Article `download()` failed with ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')) on URL https://www.makeuseof.com/generative-ai-tools-improve-work-life/


In [15]:
searched_articles_objs[0].text

"Join Fox News for access to this content Plus special access to select articles and other premium content with your account - free of charge. Please enter a valid email address.\n\nWhat do you get when you combine mobility with virtual reality?\n\nThe Honda XR Mobility Experience. It merges the physical thrill of mobility with the fantastical realms of virtual reality.\n\nThis unique blend of technology offers an unparalleled experience that transcends the boundaries of imagination.\n\nCLICK TO GET KURT’S FREE CYBERGUY NEWSLETTER WITH SECURITY ALERTS, QUICK VIDEO TIPS, TECH REVIEWS AND EASY HOW-TO’S TO MAKE YOU SMARTER\n\nWhat is the Honda Uni-One?\n\nAt the heart of this immersive experience lies the Honda Uni-One, a hands-free, personal mobility device equipped with Honda's Omni Traction Drive System. This self-stabilizing electric device offers a seamless, omnidirectional movement experience, allowing you to glide effortlessly in any direction with a simple shift of weight. A batte

Need to remove stop words when comparing the text. Need to remove \n

In [16]:
#searched_articles_objs[0].publish_date

In [17]:
import nltk
nltk.download('stopwords')


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\js774\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [18]:
import nltk
nltk.download('wordnet')


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\js774\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [19]:
import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

def preprocess_text(text):
    # Lowercasing
    text = text.lower()
    
    # Removing HTML tags, special characters, and newline characters
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = text.replace('\n', ' ')
    
    # Tokenization
    tokens = word_tokenize(text)
    
    # Removing stop words
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]
    
    # Lemmatization
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    
    return ' '.join(tokens)

In [20]:
preprocessed_searched_articles_text = []

for article in searched_articles_objs:
    preprocessed_article = preprocess_text(article.text)
    preprocessed_searched_articles_text.append(preprocessed_article)


After filtering based on topic, time, website, we only have this many articles we will compare similarity to:

In [21]:
len(preprocessed_searched_articles_text)

15

Example text after preprocessing for checking similarity:

In [22]:
preprocessed_searched_articles_text[0]

'join fox news access content plus special access select article premium content account free charge please enter valid email address get combine mobility virtual reality honda xr mobility experience merges physical thrill mobility fantastical realm virtual reality unique blend technology offer unparalleled experience transcends boundary imagination click get kurts free cyberguy newsletter security alert quick video tip tech review easy howtos make smarter honda unione heart immersive experience lie honda unione handsfree personal mobility device equipped hondas omni traction drive system selfstabilizing electric device offer seamless omnidirectional movement experience allowing glide effortlessly direction simple shift weight battery power device reach speed mph support maximum user weight pound hondas teach teen drive safely balancing promise new vr tech xray vision privacy concern xr mobility experience work donning vr headset boarding unione taken journey digital landscape serene s

We need to make the corpus articles preprocessed to compare for similarities.

In [23]:
import pandas as pd

# Load the CSV file
df = pd.read_excel('Corpus.xlsx')
#df = pd.read_csv('C:/Users/js774/News Recommendation Agent/Corpus.csv')

In [24]:
df.columns

Index(['News Link', 'Date of Article', 'Date Selected',
       'Why do you recommend it?'],
      dtype='object')

In [25]:
CorpusURLs = df['News Link'].tolist()

In [26]:
CorpusURLs

['https://techcrunch.com/2024/01/07/what-is-google-gemini-ai/',
 'https://nvidianews.nvidia.com/news/generative-ai-rtx-pcs-and-workstations',
 'https://www.nytimes.com/2024/01/08/technology/ai-robots-chatbots-2024.html',
 'https://www.cnbc.com/2024/01/08/openai-responds-to-new-york-times-lawsuit.html',
 'https://www.artificialintelligence-news.com/2024/01/08/mcafee-unveils-ai-powered-deepfake-audio-detection/',
 'https://www.axios.com/2024/01/10/walmart-app-ai-ces',
 'https://news.sap.com/2024/01/new-ai-driven-retail-capabilities-enhance-customer-experience/',
 'https://www.usatoday.com/story/tech/news/2024/01/09/walmart-ces-ai-tech/72161108007/',
 'https://www.newscientist.com/article/2412199-ai-can-tell-if-prints-from-two-different-fingers-belong-to-same-person/',
 'https://www.socialmediatoday.com/news/microsoft-launches-new-generative-ai-ad-creation-tool/704388/',
 'https://www.cnn.com/2024/01/16/tech/bill-gates-ai-gps-interview/index.html',
 'https://techcrunch.com/2024/01/16/open

In [27]:
corpus_objs, failed_corpus_articles = scrape_articles(CorpusURLs)

Error processing article at https://www.axios.com/2024/01/10/walmart-app-ai-ces: Article `download()` failed with 403 Client Error: Forbidden for url: https://www.axios.com/2024/01/10/walmart-app-ai-ces on URL https://www.axios.com/2024/01/10/walmart-app-ai-ces
Error processing article at https://venturebeat.com/ai/openai-adds-read-aloud-voiceover-to-chatgpt-allowing-it-to-speak-its-outputs/: Article `download()` failed with 403 Client Error: Forbidden for url: https://venturebeat.com/ai/openai-adds-read-aloud-voiceover-to-chatgpt-allowing-it-to-speak-its-outputs/ on URL https://venturebeat.com/ai/openai-adds-read-aloud-voiceover-to-chatgpt-allowing-it-to-speak-its-outputs/


In [28]:
failed_searched_articles

['https://www.xda-developers.com/copilot-gpt-4-turbo-model-free/',
 'https://www.makeuseof.com/generative-ai-tools-improve-work-life/']

In [29]:
failed_corpus_articles

['https://www.axios.com/2024/01/10/walmart-app-ai-ces',
 'https://venturebeat.com/ai/openai-adds-read-aloud-voiceover-to-chatgpt-allowing-it-to-speak-its-outputs/']

In [30]:
#corpus_objs[0].summary

In [31]:
preprocessed_corpus_articles_text = []
#Need to do the tokenization, Lemmatization, etc. For Corpus articles
for article in corpus_objs:
    preprocessed_article = preprocess_text(article.text)
    preprocessed_corpus_articles_text.append(preprocessed_article)

Model for recommendation, utilizes TF-IDF(Term Frequency-Inverse Document Frequency)

### How the recommendation system works:
<p>Each article (document) is represented as a TF-IDF vector.<br>
The cosine similarity is calculated between the TF-IDF vectors of the searched articles and the weighted corpus articles.<br>
The result is a similarity matrix, and the rankings are determined based on the average similarity scores.</p>

In [32]:
#You can change how important the corpus articles are compared to the searched articles for making recommendations

In [33]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def recommend_articles(corpus_articles, searched_articles):
    # Adjust the weight for corpus articles
    weighted_corpus_articles = corpus_articles * 3  # Increase the weight by repeating corpus articles

    # Combine weighted corpus articles and searched articles
    all_articles = weighted_corpus_articles + searched_articles

    # Use TfidfVectorizer to convert text into vectors
    vectorizer = TfidfVectorizer(stop_words='english')
    tfidf_matrix = vectorizer.fit_transform(all_articles)

    # Calculate cosine similarity
    similarities = cosine_similarity(tfidf_matrix)

    # Extract similarities for searched articles (excluding weighted corpus articles)
    searched_similarities = similarities[len(weighted_corpus_articles):, :len(weighted_corpus_articles)]

    # Rank searched articles by similarity to the weighted corpus articles
    ranked_articles = sorted(enumerate(searched_similarities.mean(axis=1)), key=lambda x: x[1], reverse=True)

    # Print and store the rankings in a list
    rankings = []
    for idx, similarity in ranked_articles:
        print(f"Similarity to corpus: {similarity:.4f}")
        # Print the index of preprocessed_searched_articles_text
        print("Index in preprocessed_searched_articles_text:", idx)
        print("Searched Article:", searched_articles[idx])
        print("=" * 50)
        # Append the ranking to the list
        rankings.append((idx, similarity))

    return rankings

Running the model now to recommend

In [34]:
rankings=recommend_articles(preprocessed_corpus_articles_text, preprocessed_searched_articles_text)

Similarity to corpus: 0.0676
Index in preprocessed_searched_articles_text: 14
Searched Article: apple said testing aipowered ad platform select group partner via business insider ai tool chooses place ad various app store promoted ad placement slot right seemingly used improve advertiser campaign performance app store search ad however business insider speculates technology could eventually used elsewhere apple gradually expands offering adsupported service ai ad placement isnt new invention ad space google facebook others rolled similar system last couple year little strange apple readying tool given small collection ad type right app store developer pay appear today tab search tab top search result bottom app product page company also sell advertising campaign news app stock app although much handled intermediary like nbcuniversal time course apple may roll even app store search ad slot may also preparing expand ad appear apps phone ai placement tool becomes lot relevant couple year 

In [35]:
#searched_articles_objs[2].url

In [36]:
rankings

[(14, 0.06760577662378772),
 (5, 0.0659798839072936),
 (3, 0.058012085482260875),
 (11, 0.05527095324213266),
 (2, 0.04699930845918541),
 (4, 0.040910813453237986),
 (9, 0.03978249566751805),
 (6, 0.03124338419026549),
 (12, 0.0309860941750803),
 (1, 0.029816557276154773),
 (0, 0.02917105157136824),
 (10, 0.023234021992422034),
 (8, 0.021617148022597205),
 (13, 0.01883343177442142),
 (7, 0.016668109499661087)]

In [37]:
#searched_articles_objs[7].url

In [38]:
# Check the total size of preprocessed_searched_articles_text
total_size = len(preprocessed_searched_articles_text)

# Extract URLs and summaries for the top and bottom ranked articles
top_n = 5  # Change this value as needed
bottom_n = 2  # Change this value as needed

top_ranked_info = [(searched_articles_objs[idx].url, searched_articles_objs[idx].summary) for idx, _ in rankings[:top_n]]
bottom_ranked_info = [(searched_articles_objs[idx].url, searched_articles_objs[idx].summary) for idx, _ in rankings[-bottom_n:]]

# Print URLs with rankings and summaries
if total_size <= 7:
    # If total size is 7 or less, print all URLs with summaries in order
    for idx, (url, summary) in enumerate(top_ranked_info):
        print(f"{idx + 1}. {url}\n   Summary: {summary}\n")
else:
    # Print top-ranked URLs with summaries
    print("Top Ranked URLs with Summaries:")
    for idx, (url, summary) in enumerate(top_ranked_info):
        print(f"{idx + 1}. {url}\n   Summary: {summary}\n")

    # Print ellipsis if there are more than 7 articles
    if total_size > 11:
        print("...")

    # Print bottom-ranked URLs with summaries
    print("\nBottom Ranked URLs with Summaries:")
    for idx, (url, summary) in enumerate(bottom_ranked_info):
        print(f"{total_size - bottom_n + idx + 1}. {url}\n   Summary: {summary}\n")


Top Ranked URLs with Summaries:
1. https://9to5mac.com/2024/03/11/report-apple-testing-ai-powered-ads-platform/
   Summary: Apple is said to be testing an AI-powered ads platform with a select group of partners, via Business Insider.
The AI tool chooses where to place ads in the various App Store promoted ad placement slots.
Right now, this is seemingly being used to improve advertiser campaign performance for App Store Search Ads.
AI ad placement isn’t a new invention in the ad space.
Over time, of course, Apple may roll out even more App Store Search Ads slots.

2. https://arstechnica.com/tech-policy/2024/03/nyt-disputes-openai-hacking-claim-by-pointing-to-chatgpt-bypassing-paywalls/
   Summary: OpenAI had argued that NYT allegedly made "tens of thousands of attempts to generate" supposedly "highly anomalous results" showing that ChatGPT would produce excerpts of NYT articles.
ChatGPT users bypassing paywallsAccording to the NYT's court filing, ChatGPT outputs initially only infringe

In [39]:
top_ranked_info[0][0]

'https://9to5mac.com/2024/03/11/report-apple-testing-ai-powered-ads-platform/'

Tell it which article we like and make it add it to the excel sheet which will be used to train it for the future

In [40]:
# Prompt the user for approved article numbers
approved_numbers_str = input("Enter the article numbers to approve (comma-separated): ")

# Check if the user entered anything
if approved_numbers_str.strip():
    # Convert approved numbers to a list of integers
    approved_numbers = [int(num.strip()) for num in approved_numbers_str.split(',')]

    # Extract URLs for the approved articles
    approved_urls = [top_ranked_info[idx - 1][0] for idx in approved_numbers]

    # Read the existing Corpus.xlsx file
    corpus_df = pd.read_excel('Corpus.xlsx')

    # Create a DataFrame with the approved URLs
    approved_df = pd.DataFrame({'News Link': approved_urls})

    # Concatenate the existing DataFrame with the approved DataFrame
    corpus_df = pd.concat([corpus_df, approved_df], ignore_index=True)

    # Save the updated DataFrame to Corpus.xlsx
    corpus_df.to_excel('Corpus.xlsx', index=False)

    print("Approved URLs have been added to Corpus.xlsx.")
else:
    print("No articles were approved. Corpus.xlsx remains unchanged.")

Enter the article numbers to approve (comma-separated): 1,2
Approved URLs have been added to Corpus.xlsx.
